# CartoonStyle — Kaggle T4×2 image factory

Generate **hundreds of 1920×1080 cartoon images** on Kaggle's free GPUs — with
**character/style consistency** from a LoRA you train right here.

**Pipeline:** SDXL + SDXL-Lightning (few-step distillation) → 1344×768 → upscale to 1920×1080
→ instant disk writes + auto-push to a private Kaggle Dataset (crash insurance).

| Job | Time on 2× T4 |
|---|---|
| Per image | ~2–4 s per GPU → **~1.5–2 s wall-clock** with both GPUs |
| 500 images | ~15–25 min |
| 1000 images | ~30–50 min |
| LoRA training (one-time) | ~1–1.5 h |

**Before you run:** Notebook settings → Accelerator **GPU T4 x2**, Internet **on**.
Attach a Kaggle Dataset with 30–50 style reference images (for `train` mode), and
optionally a previous outputs dataset (to `RESUME_FROM`).


In [ ]:
# ==================== CONFIG — edit me ====================
MODE = "generate"                # "generate" | "train" | "caption"

REPO_URL    = "https://github.com/AsuraXP/CartoonStyle.git"
REPO_BRANCH = "arena/22c9cd35-cartoonstyle"   # switch to "main" after merge

# --- generate ---
CONFIG_YAML     = "config/default.yaml"   # or config/paper_doodle.yaml
IMAGES          = 300        # images this run (under 1k = one short session)
SEED            = 42
BATCH           = 2          # per GPU
LIGHTNING_STEPS = 4          # 2 = fastest drafts, 4 = balanced, 8 = max quality
LORA_PATH       = ""         # e.g. "/kaggle/input/cartoonstyle-lora/final"
LORA_SCALE      = 0.85
OUT_DIR         = "/kaggle/working/cartoonstyle-out"
RESUME_FROM     = []         # e.g. ["/kaggle/input/cartoonstyle-out"]
NO_AUTO_PUSH    = False      # True = stay local, no Kaggle Dataset pushes

# --- train / caption ---
DATA_DIR       = "/kaggle/input/cartoonstyle-refs"   # 30-50 reference images
TRAIN_OUT      = "/kaggle/working/cartoonstyle-lora"
TRIGGER        = "ckstyle"  # unique token for YOUR style/character
CAPTION_PREFIX = "ckstyle, flat cartoon style, thick black outlines, stickman character"
TRAIN_STEPS    = 1200
RANK           = 16
LR             = 1e-4
# ==========================================================


In [ ]:
import os, sys, subprocess

IN_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if IN_KAGGLE else "."
REPO_DIR = os.path.join(WORK, "CartoonStyle")

def sh(cmd, **kw):
    printable = " ".join(cmd) if isinstance(cmd, list) else cmd
    print("$", printable, flush=True)
    return subprocess.run(cmd, shell=isinstance(cmd, str), **kw)

if not os.path.isdir(os.path.join(REPO_DIR, "cartoonstyle")):
    r = sh(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR],
           capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:", r.stderr.strip())
        print("trying repo zip attached as a Kaggle Dataset ...")
        import glob, zipfile, shutil
        cands = glob.glob("/kaggle/input/*/*.zip")
        if not cands:
            raise SystemExit("Could not fetch the repo. Make it public on GitHub or attach a zip dataset (see README).")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(WORK)
        extracted = [d for d in os.listdir(WORK) if os.path.isdir(os.path.join(WORK, d))]
        print("extracted:", extracted)
        os.rename(os.path.join(WORK, extracted[0]), REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

import torch
N_GPUS = max(1, torch.cuda.device_count())
print("GPUs:", N_GPUS, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), 'Turn on the GPU accelerator: Settings -> Accelerator -> GPU T4 x2'


In [ ]:
# Auto-caption reference images -> sidecar .txt files (then HAND-EDIT them!)
if MODE == "caption":
    r = sh([sys.executable, "-m", "cartoonstyle.caption",
            "--data-dir", DATA_DIR, "--prefix", CAPTION_PREFIX])
    assert r.returncode == 0


In [ ]:
# Train your style/character LoRA on both T4s (one-time, ~1-1.5h)
if MODE == "train":
    cmd = ["accelerate", "launch",
           "--num_processes", str(N_GPUS), "--mixed_precision", "fp16",
           "-m", "cartoonstyle.train_lora",
           "--data-dir", DATA_DIR, "--out-dir", TRAIN_OUT,
           "--trigger", TRIGGER,
           "--steps", str(TRAIN_STEPS), "--rank", str(RANK), "--lr", str(LR),
           "--save-every", "250"]
    r = sh(cmd)
    assert r.returncode == 0, 'training failed — see log above'
    print("LoRA saved to", os.path.join(TRAIN_OUT, "final"))


In [ ]:
# Batch-generate on both T4s — resumable, auto-pushes to a Kaggle Dataset
if MODE == "generate":
    cmd = [sys.executable, "-m", "cartoonstyle.generate",
           "--config", CONFIG_YAML,
           "--images", str(IMAGES), "--seed", str(SEED),
           "--gpus", str(N_GPUS), "--batch", str(BATCH),
           "--lightning-steps", str(LIGHTNING_STEPS),
           "--out", OUT_DIR]
    if LORA_PATH:
        cmd += ["--lora", LORA_PATH, "--lora-scale", str(LORA_SCALE)]
    for d in RESUME_FROM:
        cmd += ["--resume-from", d]
    if NO_AUTO_PUSH:
        cmd.append("--no-push")
    r = sh(cmd)
    assert r.returncode == 0, 'generation failed — completed images are safe; re-run this cell to resume'


In [ ]:
# Preview contact sheet of the newest images
if MODE == "generate":
    import glob, math
    from PIL import Image
    files = sorted(glob.glob(os.path.join(OUT_DIR, "images", "*.png")))[-24:]
    if files:
        cols, thumb = 4, (480, 270)
        rows = math.ceil(len(files) / cols)
        sheet = Image.new("RGB", (cols * thumb[0], rows * thumb[1]), "white")
        for i, f in enumerate(files):
            im = Image.open(f).convert("RGB").resize(thumb, Image.LANCZOS)
            sheet.paste(im, ((i % cols) * thumb[0], (i // cols) * thumb[1]))
        sheet.save(os.path.join(OUT_DIR, "preview.png"))
        display(sheet)
    else:
        print("no images yet")


In [ ]:
# Final snapshot + Kaggle Dataset push (safe to run any time)
if MODE == "generate":
    from cartoonstyle import kaggle_io
    files = kaggle_io.list_new_images(OUT_DIR)
    z = kaggle_io.update_zip(OUT_DIR, files)
    ok = kaggle_io.kaggle_push(OUT_DIR, "cartoonstyle-final")
    print("zip:", z)
    print("kaggle dataset push:", "ok" if ok else "skipped/failed (kaggle CLI not authed?)")
    print("images:", len(files))
